# V6.2 — E5 Stage-B + selective BGE-M3 reranking

Воспроизводимый анализ лучшего submission Даниила Жукова. Обучение не требуется: notebook использует сохранённые predictions E5/BGE, восстанавливает component-safe holdout, подбирает долю reranking и вес BGE.

**Production recipe:** E5 swap-TTA на всех парах; BGE swap-TTA только на top-10% по E5 внутри категории; локальный rank blend `0.80 E5 + 0.20 BGE`.

**Результат:** manual full `0.770112 → 0.771457`; independent eval `0.766715 → 0.768955`; Public LB `0.524467 → 0.529096`.


## Входные файлы

К notebook должны быть подключены исходные human/LLM parquet и output предыдущего scoring-запуска:

- `bge_manual.npy` — BGE для всех 365 654 human pairs;
- `e5_manual_val.npy` — E5 для 72 948 untouched human pairs;
- `bge_llm.npy`, `e5_llm.npy` — predictions exact LLM holdout из 191 555 пар.


In [ ]:
import json, gc
from pathlib import Path
import numpy as np, pandas as pd, polars as pl
from sklearn.metrics import average_precision_score

BASE=Path('/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items')
HUMAN,HI=BASE/'matches.parquet',BASE/'items_human.parquet'
LLM,LI=BASE/'matches_llm.parquet',BASE/'items.parquet'
OUT=Path('/kaggle/working/selective_e5_bge_v6'); OUT.mkdir(exist_ok=True)

def locate(name):
    for root in (Path('/kaggle/working'),Path('/kaggle/input')):
        hits=list(root.rglob(name))
        if hits:return hits[0]
    raise FileNotFoundError(name)

print('predictions:',{n:str(locate(n)) for n in ['bge_manual.npy','e5_manual_val.npy','bge_llm.npy','e5_llm.npy']})


In [ ]:
def component_mask(a,b,frac,seed):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y:parent[y]=x
    comp=np.fromiter((find(x) for x in a),np.int64,len(a))
    groups=np.unique(comp); rng=np.random.RandomState(seed)
    chosen=set(groups[rng.rand(len(groups))<frac])
    return np.fromiter((x in chosen for x in comp),bool,len(comp))

def add_category(df,items):
    need=df.select(pl.col('id1').alias('id')).unique()
    cats=(pl.scan_parquet(items).select('id','category')
          .join(need.lazy(),on='id',how='semi').collect(engine='streaming'))
    out=(df.with_row_index('_row').join(cats.rename({'id':'id1'}),on='id1',how='left')
         .sort('_row').drop('_row'))
    assert out['category'].null_count()==0
    return out

def rank(x):return pd.Series(x).rank(method='average',pct=True).to_numpy()

def selective(e5,bge,categories,top_frac,bge_weight):
    out=np.empty(len(e5),np.float64)
    for category in np.unique(categories):
        ix=np.flatnonzero(categories==category); n=len(ix)
        out[ix]=rank(e5[ix]); count=max(1,int(np.ceil(n*top_frac)))
        chosen=ix[np.argsort(e5[ix],kind='stable')[-count:]]
        mixed=(1-bge_weight)*rank(e5[chosen])+bge_weight*rank(bge[chosen])
        out[chosen]=(n-count)/n+rank(mixed)*count/n
    return out

def macro(y,p,c,mask=None):
    if mask is None:mask=np.ones(len(y),bool)
    scores=[]
    for category in np.unique(c[mask]):
        m=mask&(c==category)
        if len(np.unique(y[m]))>1:scores.append(average_precision_score(y[m],p[m]))
    return float(np.mean(scores))


## Human component-safe validation

Untouched 20% восстанавливается тем же union-find split (`seed=42`), затем ещё раз делится по компонентам на tune/eval (`seed=2026`). Параметры выбираются только по tune; eval остаётся независимой проверкой направления.


In [ ]:
bge_all=np.load(locate('bge_manual.npy'))
e5=np.load(locate('e5_manual_val.npy'))
human=pl.read_parquet(HUMAN,columns=['id1','id2','target'])
untouched=component_mask(human['id1'].to_numpy(),human['id2'].to_numpy(),.20,42)
val=add_category(human.filter(pl.Series(untouched)),HI)
bge=bge_all[untouched]
assert len(val)==len(e5)==len(bge)==72_948
y=val['target'].to_numpy().astype(np.int8)
categories=val['category'].cast(pl.String).to_numpy()
tune=component_mask(val['id1'].to_numpy(),val['id2'].to_numpy(),.50,2026)
evaluation=~tune
baseline={'tune':macro(y,e5,categories,tune),'eval':macro(y,e5,categories,evaluation),'full':macro(y,e5,categories)}
print('E5 baseline:',baseline)


In [ ]:
rows=[]
for top_frac in (.05,.10,.15,.20,.25,.30):
    for bge_weight in (0,.025,.05,.075,.10,.125,.15,.20):
        prediction=selective(e5,bge,categories,top_frac,bge_weight)
        rows.append({'top_frac':top_frac,'bge_weight':bge_weight,
                     'tune':macro(y,prediction,categories,tune),
                     'eval':macro(y,prediction,categories,evaluation),
                     'full':macro(y,prediction,categories)})
grid=pd.DataFrame(rows).sort_values('tune',ascending=False)
display(grid.head(12))
grid.to_csv(OUT/'grid.csv',index=False)


## Production choice

Validation-optimum `top=30%, BGE=0.15` не прошёл лимит времени. Сабмит использует `top=10%, BGE=0.20`: он сохраняет независимый прирост на eval и сокращает BGE-инференс втрое.


In [ ]:
TOP_FRAC,BGE_WEIGHT=.10,.20
deployed=grid[np.isclose(grid.top_frac,TOP_FRAC)&np.isclose(grid.bge_weight,BGE_WEIGHT)].iloc[0].to_dict()
print('deployed:',deployed)
print('eval delta:',deployed['eval']-baseline['eval'])
print('full delta:',deployed['full']-baseline['full'])
llm_result=None


## Exact LLM holdout

Этот шаг нужен как дополнительная диагностика domain shift. Финальный production-рецепт выбирается по human validation, а не по LLM holdout.


In [ ]:
bge_llm=np.load(locate('bge_llm.npy')); e5_llm=np.load(locate('e5_llm.npy'))
llm=pl.read_parquet(LLM,columns=['id1','id2','target'])
llm_mask=component_mask(llm['id1'].to_numpy(),llm['id2'].to_numpy(),.03,13)
llm_val=(llm.filter(pl.Series(llm_mask))
          .filter((pl.col('target')<=.2)|(pl.col('target')>=.8))
          .with_columns((pl.col('target')>=.5).cast(pl.Int8).alias('target')))
llm_val=add_category(llm_val,LI)
assert len(llm_val)==len(e5_llm)==len(bge_llm)==191_555
yl=llm_val['target'].to_numpy(); cl=llm_val['category'].cast(pl.String).to_numpy()
pllm=selective(e5_llm,bge_llm,cl,TOP_FRAC,BGE_WEIGHT)
llm_result={'e5':macro(yl,e5_llm,cl),'selective':macro(yl,pllm,cl)}
print('LLM:',llm_result)
del llm; gc.collect()


In [ ]:
result={
    'version':'V6.2',
    'selected_top_frac':TOP_FRAC,
    'selected_e5_weight':1-BGE_WEIGHT,
    'selected_bge_weight':BGE_WEIGHT,
    'manual_e5':baseline,
    'manual_selective':{k:float(deployed[k]) for k in ('tune','eval','full')},
    'llm':llm_result,
    'previous_public_lb':0.5244670740165974,
    'public_lb':0.5290960422265529,
    'training_required':False
}
(OUT/'selective_config.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print(json.dumps(result,ensure_ascii=False,indent=2))
print('saved:',sorted(p.name for p in OUT.iterdir()))


## Submission result

Прошедший production container получил Public LB **0.5290960422**, улучшив предыдущий результат **0.5244670740** на **0.0046289682**. Веса моделей и submission ZIP намеренно не хранятся в Git.
